# Parkinson's Disease — Model Training Pipeline
In this notebook, we train 5 machine learning models on acoustic vocal features:
1. Linear SVM
2. RBF SVM
3. Decision Tree
4. Random Forest
5. K-Nearest Neighbors (KNN)

In [1]:
import os
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

MODEL_DIR = "saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

## 1. Load Data and Split Features & Labels
Split into 80% training data and 20% testing data.

In [2]:
# Load cleaned Parkinson's dataset
df = pd.read_csv(os.path.join("processed_data", "cleaned_parkinsons.csv"))

# Separate features (X) and target (y)
X = df.drop(columns=['status'])
y = df['status']

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}, y_test shape:  {y_test.shape}")

X_train shape: (960, 22), y_train shape: (960,)
X_test shape:  (240, 22), y_test shape:  (240,)


## 2. Feature Scaling (StandardScaler)
Standardize continuous acoustic features and save `parkinsons_scaler.pkl`.

In [3]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler
joblib.dump(scaler, os.path.join(MODEL_DIR, "parkinsons_scaler.pkl"))
print("[OK] Saved scaler to saved_models/parkinsons_scaler.pkl")

[OK] Saved scaler to saved_models/parkinsons_scaler.pkl


## 3. Train and Save Models
- Linear SVM, RBF SVM, and KNN use **scaled features**.
- Decision Tree and Random Forest use **raw features**.

In [4]:
models = {
    'parkinsons_linear_svm.pkl': (SVC(kernel='linear', probability=True, random_state=42), True),
    'parkinsons_rbf_svm.pkl': (SVC(kernel='rbf', probability=True, random_state=42), True),
    'parkinsons_decision_tree.pkl': (DecisionTreeClassifier(max_depth=4, random_state=42), False),
    'parkinsons_random_forest.pkl': (RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42), False),
    'parkinsons_knn.pkl': (KNeighborsClassifier(n_neighbors=5), True)
}

for filename, (model, use_scaled) in models.items():
    X_train_data = X_train_scaled if use_scaled else X_train
    model.fit(X_train_data, y_train)
    save_path = os.path.join(MODEL_DIR, filename)
    joblib.dump(model, save_path)
    print(f"[OK] Trained & Saved: {filename}")

[OK] Trained & Saved: parkinsons_linear_svm.pkl
[OK] Trained & Saved: parkinsons_rbf_svm.pkl
[OK] Trained & Saved: parkinsons_decision_tree.pkl


[OK] Trained & Saved: parkinsons_random_forest.pkl
[OK] Trained & Saved: parkinsons_knn.pkl


## 4. Verification
Check all files saved in `saved_models/`.

In [5]:
print("Parkinson's model files:")
for f in sorted(os.listdir(MODEL_DIR)):
    if f.startswith("parkinsons_"):
        print(" -", f)

Parkinson's model files:
 - parkinsons_decision_tree.pkl
 - parkinsons_knn.pkl
 - parkinsons_linear_svm.pkl
 - parkinsons_random_forest.pkl
 - parkinsons_rbf_svm.pkl
 - parkinsons_scaler.pkl
